# Prefix-cache mutation benchmark quick test

This notebook loads a small dataset shard, applies one mutation, and inspects overlap metrics.

In [ ]:
from data_loader import DataLoadConfig, load_examples
from prompt_generator import AlgorithmicMeaningPreservingMutator, AlgorithmicMeaningChangingMutator
from overlap_analyzer import OverlapAnalyzer
from llm_fn import LLMConfig, make_llm_fn
from prompt_generator import save_prompt_records

In [ ]:
# RAG example: pick a dataset that has question + context-like fields
cfg = DataLoadConfig(
    workload='scientific',
    dataset_name=None,
    split='train',
    max_samples=3,
    shard_index=0,
    num_shards=1,
)
examples = load_examples(cfg)
len(examples), examples[0]

In [ ]:
mutator = AlgorithmicMeaningChangingMutator(seed=42)
record = mutator.mutate_scientific(examples[0], mutation_type='parameter_change', mutation_severity=1.0)
print(record.base_prompt)
print('\n' + '-'*80 + '\n')
print(record.mutated_prompt)

In [ ]:
analyzer = OverlapAnalyzer(semantic_model_name=None)
metrics = analyzer.analyze(record.base_prompt, record.mutated_prompt)
metrics

In [ ]:
# Optional LLM-generated test with mock backend
llm_fn = make_llm_fn(LLMConfig(backend='mock', model='mock-model'))
llm_fn('Rewrite the following user query so that it preserves the exact meaning and expected answer, but changes the wording.\n\nQuery: What is the capital of France?')